<a href="https://colab.research.google.com/github/selaluar/2411533008_Big_Data_b/blob/main/Praktikum/BD_B_P02_2411533008_Arya_Pratama_Hendri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 2: Pengumpulan dan Pra-pemrosesan Data

Mata kuliah:Praktikum Big Data  
Nama:Arya Pratama Hendri  
NIM:2411533008  
Kelas:B  


Notebook ini berisi proses pembuatan dan pembersihan data transaksi marketplace sintetis. Seluruh data dibuat untuk keperluan praktikum dan tidak berasal dari transaksi pelanggan sebenarnya.

## CELL 1 - Install Faker

**Fungsi kode:**

Perintah `pip install` memasang library Faker pada Google Colab. Library ini dipakai untuk membuat nama pelanggan, nama produk, kota, dan tanggal secara otomatis. Opsi `-q` membuat proses instalasi tampil lebih ringkas.

In [13]:
!pip install faker -q

## CELL 2 - K-1. Import Library dan Inisialisasi

**Fungsi kode:**

Cell ini memanggil library yang dipakai selama praktikum:

- NumPy membantu menangani nilai kosong dalam bentuk `NaN`.
- pandas mengolah data berbentuk tabel atau DataFrame.
- Faker menghasilkan data contoh seperti nama dan kota.
- random memilih nilai secara acak ketika dataset dibuat.

Versi pandas ikut ditampilkan supaya lingkungan yang dipakai dapat diperiksa jika muncul perbedaan hasil.

In [14]:
import numpy as np
import pandas as pd
from faker import Faker
import random

print("Versi pandas:", pd.__version__)

Versi pandas: 2.2.3


## CELL 3 - K-2. Membuat Dataset Sintetis

**Fungsi kode:**

Cell ini membuat 500 transaksi marketplace dengan `SEED = 42`. Seed menjaga hasil acak tetap konsisten saat notebook dijalankan ulang.

Setiap transaksi memiliki ID, nama pelanggan, produk, kategori, harga, jumlah barang, metode pembayaran, tanggal, kota pengiriman, dan rating. Beberapa nilai sengaja dibuat tidak rapi, misalnya format harga dan tanggal yang berbeda, tulisan kategori yang tidak seragam, serta data yang kosong.

Kode lalu menyalin 15 baris sebagai duplicate. Karena itu, `transaksi_mentah.csv` berisi 515 baris. Pengacakan urutan baris membuat data duplicate tidak berkumpul di satu bagian tabel.

In [15]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = [
    "Elektronik",
    "Fashion",
    "Kesehatan",
    "Rumah Tangga",
    "Olahraga",
    "Buku",
]
metode_bayar = [
    "Transfer Bank",
    "E-Wallet",
    "COD",
    "Kartu Kredit",
]

rows = []

for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(
        ["Pro", "Lite", "Max", "Basic", ""]
    )
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice(
        [15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000]
    )
    qty = random.randint(1, 5)

    # Variasi format harga
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [
        tgl.strftime("%Y-%m-%d"),
        tgl.strftime("%d/%m/%Y"),
        tgl.strftime("%d-%m-%Y"),
    ]
    tanggal = random.choice(tgl_variants)

    # Variasi metode pembayaran dan kapitalisasi kategori
    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append(
        {
            "transaction_id": trx_id,
            "customer_name": nama_pelanggan,
            "product_name": produk.strip(),
            "category": kategori,
            "price": harga,
            "quantity": qty,
            "payment_method": metode,
            "transaction_date": tanggal,
            "shipping_city": kota,
            "rating": rating,
        }
    )

df = pd.DataFrame(rows)

# Menyisipkan missing value
for col, frac in [
    ("customer_name", 0.02),
    ("shipping_city", 0.03),
    ("payment_method", 0.015),
]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Menambahkan 15 baris duplicate
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)

# Mengacak urutan data
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Menyimpan data mentah
df.to_csv("transaksi_mentah.csv", index=False)

print("Jumlah baris:", len(df))
df.head()

Jumlah baris: 515


,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000,4,kartu kredit,2026-07-17,Blitar,1.0
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,Rp500.000,1,E-Wallet,2026-07-13,Lubuklinggau,NaN
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000,1,COD,2026-08-29,Probolinggo,2.0
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,Rp250.000,1,COD,31/07/2026,Tual,4.0
4,TRX00074,NaN,Eius,Olahraga,250000.0,4,NaN,05/07/2026,NaN,1.0


## CELL 4 - K-3. Mengecek Missing Value

**Fungsi kode:**

`df.isnull().sum()` menghitung jumlah data kosong pada setiap kolom. Hasilnya dipakai untuk menentukan kolom mana yang perlu dibuang, diisi, atau dibiarkan kosong.

Pada dataset ini, `customer_name`, `payment_method`, dan `shipping_city` memiliki data kosong. Kolom `rating` juga memiliki nilai kosong karena pelanggan tidak wajib memberi rating.

In [16]:
print("Jumlah missing value per kolom:")
print(df.isnull().sum())

Jumlah missing value per kolom:
transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


## CELL 5 - K-4. Mendeteksi dan Menghapus Duplicate

**Fungsi kode:**

Pemeriksaan pertama menghitung baris yang seluruh isinya sama. Pemeriksaan kedua melihat pengulangan pada `transaction_id`. Keduanya menghasilkan 15 duplicate.

`drop_duplicates()` menghapus salinan tambahan dan mempertahankan satu baris dari setiap transaksi. Jumlah data turun dari 515 menjadi 500 baris. `.copy()` membuat DataFrame hasil pembersihan dapat diubah tanpa peringatan dari pandas.

In [17]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df["transaction_id"].duplicated().sum())

df = df.drop_duplicates().copy()

print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 15
transaction_id duplicate: 15
Jumlah baris setelah drop_duplicates(): 500


## CELL 6 - K-3. Menangani Missing Value

**Fungsi kode:**

`dropna()` membuang baris yang tidak memiliki `customer_name` atau `payment_method` karena kedua data tersebut dibutuhkan untuk mengenali transaksi. Setelah proses ini, jumlah data menjadi 490 baris.

Nilai kosong pada `shipping_city` diganti dengan `Tidak Diketahui` memakai `fillna()`. Baris tersebut tetap berguna meskipun nama kotanya tidak tercatat. Rating dibiarkan `NaN` karena kolom itu bersifat opsional.

In [18]:
df = df.dropna(subset=["customer_name", "payment_method"]).copy()
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print("Jumlah baris setelah menangani missing value:", len(df))
print("\nMissing value setelah penanganan:")
print(df.isnull().sum())

Jumlah baris setelah menangani missing value: 490

Missing value setelah penanganan:
transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              157
dtype: int64


## CELL 7 - K-5a. Standardisasi Teks

**Fungsi kode:**

Cell ini merapikan isi kolom `category`, `payment_method`, dan `shipping_city`. Data diubah ke tipe string, spasi di awal dan akhir dibuang dengan `str.strip()`, lalu penulisan huruf diseragamkan memakai `str.title()`.

Hasil `str.title()` mengubah `COD` menjadi `Cod`, sehingga kode menggantinya kembali menjadi `COD`. Daftar kategori dan metode pembayaran kemudian ditampilkan untuk memastikan variasinya sudah konsisten.

In [19]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# COD adalah singkatan, jadi dikembalikan ke huruf kapital penuh
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

print("Category:", sorted(df["category"].unique()))
print("Payment method:", sorted(df["payment_method"].unique()))

Category: ['Buku', 'Elektronik', 'Fashion', 'Kesehatan', 'Olahraga', 'Rumah Tangga']
Payment method: ['COD', 'E-Wallet', 'Kartu Kredit', 'Transfer Bank']


## CELL 8 - K-5b. Membersihkan Kolom Price

**Fungsi kode:**

Fungsi `bersihkan_harga()` mengubah berbagai bentuk penulisan harga menjadi angka. Kode menghapus spasi dan simbol `Rp`, membedakan akhiran desimal `.0` dari titik pemisah ribuan, lalu mengubah hasilnya menjadi `float`.

Jika sebuah nilai tidak dapat dikonversi, fungsi mengembalikan `NaN` agar kesalahannya mudah ditemukan. Rentang harga ikut ditampilkan untuk mengecek apakah hasil konversinya masuk akal.

In [20]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan

    nilai = str(x).strip().replace("Rp", "")

    # Pada dataset ini, ".0" adalah akhiran desimal.
    # Titik lainnya adalah pemisah ribuan Indonesia.
    if nilai.endswith(".0") and nilai[:-2].isdigit():
        nilai = nilai[:-2]
    else:
        nilai = nilai.replace(".", "")

    nilai = nilai.replace(",", ".")

    try:
        return float(nilai)
    except ValueError:
        return np.nan


df["price"] = df["price"].apply(bersihkan_harga)

print("Tipe data price:", df["price"].dtype)
print("Rentang harga:", df["price"].min(), "sampai", df["price"].max())

Tipe data price: float64
Rentang harga: 15000.0 sampai 1200000.0


## CELL 9 - K-5c. Standardisasi Tanggal

**Fungsi kode:**

Dataset mentah memakai tiga pola tanggal: `YYYY-MM-DD`, `DD/MM/YYYY`, dan `DD-MM-YYYY`. Fungsi `parse_tanggal()` mencoba ketiga pola tersebut satu per satu pada setiap nilai.

Tanggal yang berhasil dibaca ditulis ulang dalam format `YYYY-MM-DD`. Jika tidak ada pola yang sesuai, fungsi memberi nilai `NaT` sebagai tanda bahwa tanggal gagal diproses. Cara ini mencegah tanggal ISO tertukar antara posisi bulan dan hari.

In [21]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT


df["transaction_date"] = (
    df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
)

print(df["transaction_date"].head(10))

0     2026-07-17
1     2026-07-13
2     2026-08-29
3     2026-07-31
5     2026-09-19
6     2026-08-11
7     2026-08-21
8     2026-08-22
9     2026-09-15
10    2026-06-22
Name: transaction_date, dtype: object


## CELL 10 - K-5d. Finalisasi Tipe Data

**Fungsi kode:**

Kolom `quantity` diubah menjadi `int` karena jumlah barang berupa bilangan bulat. Kolom `price` ditetapkan sebagai `float` supaya dapat dipakai untuk perhitungan. `df.dtypes` menampilkan tipe semua kolom setelah proses konversi.

In [22]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

print(df.dtypes)

transaction_id              object
customer_name               object
product_name                object
category            string[python]
price                      float64
quantity                     int64
payment_method      string[python]
transaction_date            object
shipping_city       string[python]
rating                     float64
dtype: object


## CELL 11 - Pemeriksaan Hasil Akhir

**Fungsi kode:**

Cell ini memeriksa hasil cleaning sebelum data disimpan. Informasi yang dicek mencakup jumlah baris, jumlah kategori, jumlah metode pembayaran, tipe data harga dan jumlah barang, sisa duplicate, serta missing value.

Perintah `assert` bertindak sebagai pemeriksaan otomatis. Notebook akan menampilkan error jika hasilnya tidak sesuai target modul, misalnya jumlah baris bukan 490 atau masih ada duplicate. Pesan `Semua pemeriksaan berhasil` muncul ketika seluruh syarat terpenuhi.

In [23]:
print("=== PEMERIKSAAN DATA ===")
print("Jumlah baris:", len(df))
print("Jumlah kategori:", df["category"].nunique())
print("Jumlah metode pembayaran:", df["payment_method"].nunique())
print("Tipe price:", df["price"].dtype)
print("Tipe quantity:", df["quantity"].dtype)
print("Duplicate tersisa:", df.duplicated().sum())
print("\nMissing value:")
print(df.isnull().sum())

assert len(df) == 490
assert df["category"].nunique() == 6
assert df["payment_method"].nunique() == 4
assert df.duplicated().sum() == 0
assert df["price"].notna().all() and (df["price"] > 0).all()
assert df["transaction_date"].notna().all()

print("\nSemua pemeriksaan berhasil.")

=== PEMERIKSAAN DATA ===
Jumlah baris: 490
Jumlah kategori: 6
Jumlah metode pembayaran: 4
Tipe price: float64
Tipe quantity: int64
Duplicate tersisa: 0

Missing value:
transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              157
dtype: int64

Semua pemeriksaan berhasil.


## CELL 12 - K-6. Ekspor Dataset Bersih

**Fungsi kode:**

`to_csv()` menyimpan DataFrame yang sudah dibersihkan ke file `transaksi_bersih.csv`. Parameter `index=False` mencegah nomor indeks pandas ikut menjadi kolom di dalam file.

Nama file mengikuti ketentuan modul karena dataset ini akan dipakai kembali pada praktikum berikutnya. Output terakhir menampilkan jumlah data yang tersimpan, yaitu 490 baris.

In [24]:
df.to_csv("transaksi_bersih.csv", index=False)

print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris
